# **Actividad 2: Experimentación con Generación de Texto sin Entrenamiento**


En esta actividad exploraremos cómo los modelos de Transformers pueden generar texto sin necesidad de entrenarlos (Fine-Tuning). Vamos a modificar los hiperparámetros de generación y compararemos los resultados con información obtenida en tiempo real.

## **1. Introducción**
- Los Transformers pueden generar texto basándose en **modelos preentrenados**.
- Ajustar **parámetros de generación** como `temperature`, `top_k` y `top_p` cambia la creatividad del texto.
- Compararemos la generación de texto con una consulta de información en línea (**simulación de RAG**).

## **2.Configuración del Entorno**

Ejecuta la siguiente celda para instalar las librerías necesarias.

In [3]:
!pip install transformers datasets torch requests

## **3. Cargar un Modelo Preentrenado sin Entrenamiento**
Vamos a cargar un modelo de lenguaje preentrenado en español y su tokenizador para generar texto.

In [4]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch

# Verificar GPU
device = 'cpu'

# Cargar modelo y tokenizador
model_name = 'datificate/gpt2-small-spanish'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Crear pipeline de generación de texto
generator = pipeline('text-generation', model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: datificate/gpt2-small-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## **4. Experimentar con Generación de Texto**
Vamos a modificar los hiperparámetros y ver cómo afectan la creatividad y coherencia del texto generado.

In [5]:
def generate_text(prompt, max_length=100, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.2):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    outputs = model.generate(
        **inputs, max_length=max_length, temperature=temperature,
        top_k=top_k, top_p=top_p, repetition_penalty=repetition_penalty, do_sample=True, pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Prueba con diferentes prompts
prompts = [
    'La inteligencia artificial en el futuro',
    'El impacto de la IA en la educación',
    'Las ventajas y riesgos de los modelos de lenguaje'
]

for prompt in prompts:
    print(f'\n🔹 Prompt: {prompt}')
    print(generate_text(prompt))


🔹 Prompt: La inteligencia artificial en el futuro
La inteligencia artificial en el futuro es una técnica de ingeniería que permite a los investigadores crear computadoras inteligentes y humanos. La inteligencia artificial ha sido desarrollada por la Universidad de California, Berkeley (UCB), que está desarrollando un sistema de inteligencia artificial para el uso académico y científico del futuro.

El proyecto de investigación de la UCB fue anunciado oficialmente en junio de 2007 con una conferencia pública sobre ciencia y tecnología de "The First Date" ("Los siete principios"). La primera fase de la investigación comenzó

🔹 Prompt: El impacto de la IA en la educación
El impacto de la IA en la educación ha sido muy grande. En el curso 2012-2013, un estudio del Instituto de Estudios Avanzados (IMEA) concluyó que el impacto más importante para la salud es la influencia positiva sobre las habilidades de aprendizaje y la inteligencia emocional; sin embargo, la investigación reciente muest

## **5. Haz tres experimentos modificando  temperature**
A partir de los 3 prompts realiza los experimientos. Es muy importante la conclusión.

In [6]:
temperatures = [0.2, 0.7, 1.2]

for temp in temperatures:
    print("\n==============================")
    print(f"TEMPERATURE = {temp}")
    print("==============================")

    for prompt in prompts:
        print(f"\n🔹 Prompt: {prompt}")
        print(generate_text(prompt, temperature=temp))


TEMPERATURE = 0.2

🔹 Prompt: La inteligencia artificial en el futuro
La inteligencia artificial en el futuro, y la tecnología de combate que se está desarrollando para el combate aéreo.

En el año 2015, "The New York Times" informó sobre un nuevo proyecto llamado ""The Battlefield"", una nueva versión del juego de estrategia de combate desarrollado por Microsoft. El juego se centra principalmente en el combate cuerpo a cuerpo con armas de fuego como los misiles tierra-aire, misiles aire-tierra, misiles aire-superficie, misiles aire-aire, misiles aire-

🔹 Prompt: El impacto de la IA en la educación
El impacto de la IA en la educación y el desarrollo del conocimiento, así como su influencia sobre las decisiones políticas y sociales.

En el ámbito académico, se ha desarrollado un amplio conocimiento científico que incluye estudios de los procesos cognitivos, sociales e ideológicos relacionados con la interacción entre humanos (por ejemplo, la psicología cognitiva). En este sentido, se ha

# Conclusiones — Temperature

### Lo que muestran tus resultados

**Temperature = 0.2**
* Texto muy conservador y técnico
* Frases largas pero poco creativas
* Se mantiene en temas muy “seguros” (educación, investigación, tecnología)
* Ejemplo claro: repite estructuras tipo *“El impacto de la IA…”*

El modelo casi no arriesga.

**Temperature = 0.7**
* Más variación en ideas
* Aparecen conceptos nuevos (máquinas, aprendizaje, sistemas)
* Sigue siendo bastante coherente

Punto de equilibrio claro.

**Temperature = 1.2**
* Aparecen frases raras o incoherentes
  * *“neuromodal de origen cerealista”*
* Cambios bruscos de tema
* Mezcla política, ciencia, educación sin conexión

El modelo empieza a “fantasear”.

### CONCLUSIÓN TEMPERATURE

La temperatura controla la **creatividad vs coherencia**:

| Valor | Comportamiento            |
| ----- | ------------------------- |
| Baja  | Texto seguro y repetitivo |
| Media | Mejor equilibrio          |
| Alta  | Creativo pero incoherente |

**Mejor valor observado: 0.7**

## **6. Haz tres experimentos modificando top_k**
A partir de los 3 prompts realiza los experimientos. Es muy importante la conclusión.

In [7]:
top_ks = [10, 50, 100]

for k in top_ks:
    print("\n==============================")
    print(f"TOP_K = {k}")
    print("==============================")

    for prompt in prompts:
        print(f"\n🔹 Prompt: {prompt}")
        print(generate_text(prompt, top_k=k))


TOP_K = 10

🔹 Prompt: La inteligencia artificial en el futuro
La inteligencia artificial en el futuro es una técnica que permite a los seres humanos manipular la información de forma remota. Los seres humanos utilizan las técnicas para crear ilusiones y manipular eventos, ya sea físicos o químicos como la percepción del tiempo o la percepción del espacio exterior (es decir, los objetos) con fines científicos; sin embargo, no todos los humanos son capaces de crear ilusiones debido al uso excesivo de estas máquinas. 

Los humanos tienen muchas posibilidades sobre cómo utilizar la tecnología, pero se basan principalmente en

🔹 Prompt: El impacto de la IA en la educación
El impacto de la IA en la educación, el acceso a los medios y las prácticas sociales se vieron alterados por la aplicación de tecnologías emergentes. La evolución del conocimiento fue un proceso que se aceleró con respecto al uso de nuevas tecnologías como el internet o la televisión digital terrestre (conectado entre 199

# Conclusiones — top_k

### Resultados

**top_k = 10**
* Frases muy parecidas entre prompts
* Mucha repetición de estructuras:
  * “La inteligencia artificial en el futuro es…”
* Poco vocabulario.

Muy restrictivo.

**top_k = 50**
* Más variedad temática
* Aparecen política, ejército, programación
* Texto más natural.

Buen equilibrio.

**top_k = 100**
* Más diversidad de ideas
* Aparecen temas nuevos (UE, OTAN, I+D)
* Ligera pérdida de coherencia.

### CONCLUSIÓN TOP_K

Controla el **tamaño del vocabulario posible**.

| Valor | Efecto                      |
| ----- | --------------------------- |
| Bajo  | Repetitivo                  |
| Medio | Natural                     |
| Alto  | Más creativo pero inestable |

**Valor óptimo observado: 50**

## **7. Haz tres experimentos modificando top_p**
A partir de los 3 prompts realiza los experimientos. Es muy importante la conclusión.

In [8]:
top_ps = [0.5, 0.9, 0.95]

for p in top_ps:
    print("\n==============================")
    print(f"TOP_P = {p}")
    print("==============================")

    for prompt in prompts:
        print(f"\n🔹 Prompt: {prompt}")
        print(generate_text(prompt, top_p=p))


TOP_P = 0.5

🔹 Prompt: La inteligencia artificial en el futuro
La inteligencia artificial en el futuro es una de las características más importantes del cerebro humano. La inteligencia artificial se basa principalmente en la capacidad de generar un conjunto de habilidades que pueden ser utilizadas para mejorar su comportamiento, y que son útiles como herramientas para mejorar sus capacidades cognitivas o para mejorar los resultados obtenidos por otros investigadores.

Los científicos han desarrollado métodos para detectar y medir inteligencia artificial utilizando técnicas de análisis de datos (como el procesamiento de información) con el fin de obtener mejores resultados sobre la inteligencia artificial humana

🔹 Prompt: El impacto de la IA en la educación
El impacto de la IA en la educación es enorme. El sistema educativo está basado principalmente sobre el aprendizaje y las habilidades que se pueden adquirir a través del uso de tecnologías como la radio, la televisión o Internet (p

# Conclusiones — top_p

### Resultados

**top_p = 0.5**
* Muy conservador
* Frases muy genéricas
* Repetición de conceptos (educación, aprendizaje).

**top_p = 0.9**
* Texto equilibrado
* Buen flujo de ideas
* Mejor coherencia general.

**top_p = 0.95**
* Más creatividad
* Aparecen errores conceptuales:
  * *“inteligencia artificial desarrollada por científicos rusos”*
* Mezcla temas sin relación.

### CONCLUSIÓN TOP_P

Controla la **diversidad probabilística**.

| Valor | Resultado                  |
| ----- | -------------------------- |
| Bajo  | Seguro                     |
| Medio | Óptimo                     |
| Alto  | Creativo pero menos fiable |

**Valor óptimo observado: 0.9**

## **8. Haz tres experimentos modificando repetition_penalty**
A partir de los 3 prompts realiza los experimientos. Es muy importante la conclusión.

In [9]:
penalties = [1.0, 1.2, 1.5]

for pen in penalties:
    print("\n==============================")
    print(f"REPETITION PENALTY = {pen}")
    print("==============================")

    for prompt in prompts:
        print(f"\n🔹 Prompt: {prompt}")
        print(generate_text(prompt, repetition_penalty=pen))


REPETITION PENALTY = 1.0

🔹 Prompt: La inteligencia artificial en el futuro
La inteligencia artificial en el futuro es el resultado de un diseño de inteligencia artificial que se ha construido a partir de la inteligencia artificial en el futuro.

La inteligencia artificial en el futuro puede ser llevada a cabo por científicos y ingenieros que trabajan con las fuerzas armadas. Los científicos y ingenieros pueden trabajar en un ambiente de combate de larga duración, con una gran capacidad de combate de gran alcance. Los ingenieros pueden usar armas de gran alcance, armas nucleares, misiles, armas de precisión, misiles y armas de

🔹 Prompt: El impacto de la IA en la educación
El impacto de la IA en la educación se produce en la cultura y en la sociedad. Se puede decir que la IA no es una herramienta de aprendizaje y que la IA no es un objetivo.

El impacto de la IA en la educación se produce en la cultura y en la sociedad. Se puede decir que la IA no es una herramienta de aprendizaje y q

# Conclusiones — repetition_penalty

### Resultados

**Penalty = 1.0**
* Muchísima repetición:
  * *“El impacto de la IA en la educación…”* repetido literalmente.
* GPT-2 muestra su problema típico.

**Penalty = 1.2**
* Desaparece la repetición excesiva
* Texto más fluido y variado.

Mejora clara.

**Penalty = 1.5**
* El modelo evita repetir demasiado…
* Pero genera frases raras o cortadas:
  * *“La inteligencia artificial en el futuro.”* (frase incompleta)

Penalización excesiva.

### CONCLUSIÓN REPETITION PENALTY

Controla la **repetición de frases**.

| Valor | Resultado     |
| ----- | ------------- |
| Bajo  | Repetitivo    |
| Medio | Óptimo        |
| Alto  | Texto forzado |

**Valor óptimo observado: 1.2**

## **9. Haz dos experimentos modificando el modelo**

*  "bigscience/bloom-560m" (más potente)
*  "PlanTL-GOB-ES/gpt2-spanish" (ajustado para español)

A partir de los 3 prompts realiza los experimientos. Es muy importante la conclusión.

In [15]:
model_name = "bigscience/bloom-560m"

tokenizer_bloom = AutoTokenizer.from_pretrained(model_name)
model_bloom = AutoModelForCausalLM.from_pretrained(model_name).to(device)

def generate_bloom(prompt):
    inputs = tokenizer_bloom(prompt, return_tensors="pt").to(device)
    outputs = model_bloom.generate(
        **inputs,
        max_length=100,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.2,
        do_sample=True,
        pad_token_id=tokenizer_bloom.eos_token_id
    )
    return tokenizer_bloom.decode(outputs[0], skip_special_tokens=True)

print("===== BLOOM RESULTS =====")
for prompt in prompts:
    print(f"\n🔹 Prompt: {prompt}")
    print(generate_bloom(prompt))

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

===== BLOOM RESULTS =====

🔹 Prompt: La inteligencia artificial en España ha
La inteligencia artificial en España ha evolucionado de una forma dramática. Pero la realidad es que aún hay mucho camino por recorrer.
El Gobierno se está moviendo para encontrar soluciones a los problemas, y lo hace con un gran interés: el sector público tiene unos recursos muy limitados pero requiere inversiones masivas e infraestructuras adecuadas (la mitad del PIB español). La inversión pública no puede ser tan solo política sino social también; debe incluir todos aquellos aspectos relacionados directamente o indirectamente como son las políticas sociales dirigidas al desarrollo humano


# Conclusiones — Cambio de modelo

## BLOOM-560M

Observado en tu output:
* Texto más largo
* Más estructura narrativa
* Más contexto (Gobierno, sector, evolución)
* Ideas más “humanas”

Mejor capacidad de razonamiento.

**Conclusión:**
Modelo más grande ⇒ mejor calidad general.

---



In [14]:
# Determinar el dispositivo
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Usar el repositorio correcto de DeepESP
model_name = "DeepESP/gpt2-spanish"

# 2. Cargar tokenizador y modelo
tokenizer_es = AutoTokenizer.from_pretrained(model_name)
model_es = AutoModelForCausalLM.from_pretrained(model_name).to(device)

def generate_es(prompt):
    inputs = tokenizer_es(prompt, return_tensors="pt").to(device)
    outputs = model_es.generate(
        **inputs,
        max_length=100,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.2,
        do_sample=True,
        pad_token_id=tokenizer_es.eos_token_id
    )
    return tokenizer_es.decode(outputs[0], skip_special_tokens=True)

prompts = ["La inteligencia artificial en España ha"]

print("===== GPT2-SPANISH RESULTS =====")
for prompt in prompts:
    print(f"\n🔹 Prompt: {prompt}")
    print(generate_es(prompt))

config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/261M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: DeepESP/gpt2-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


===== GPT2-SPANISH RESULTS =====

🔹 Prompt: La inteligencia artificial en España ha
La inteligencia artificial en España ha sido la clave para comprender que las empresas, y no sólo los países con sus grandes recursos económicos, están sujetas a un fuerte número de personas; es una realidad que se basa en el modelo económico y social más avanzado del siglo XX. 

¿Qué son las empresas? , ¿qué sucede en este mundo tan complicado? Hay mucha gente muy importante y muchas cosas nuevas sobre ellas que no van a afectar al resto de su vida. Por ejemplo, las sociedades humanas han


## GPT2 Spanish (DeepESP)

Observado:
* Español más natural
* Mejor gramática
* Más estilo conversacional

Menos técnico pero más fluido.

**Conclusión:**
Modelo especializado en español ⇒ mejor calidad lingüística.

---

# CONCLUSIÓN FINAL GLOBAL

Los experimentos demuestran que:

### Los hiperparámetros controlan el comportamiento del modelo sin entrenarlo

Podemos ajustar:
* Creatividad → temperature
* Diversidad → top_k / top_p
* Repetición → repetition_penalty

Es posible mejorar mucho la calidad **sin fine-tuning**.

### Configuración óptima encontrada

Basado en los resultados:

```python
temperature = 0.7
top_k = 50
top_p = 0.9
repetition_penalty = 1.2